# OpenNyAI rhetorical-role tagging — 100 test judgments

This notebook runs OpenNyAI's **pretrained** rhetorical-role tagger over the 100 non-empty test judgments in `IN-Abs-prepared/test/judgement`.

It writes resumable JSONL checkpoints to `opennyai_test_cpu_100/`.


### Setup
Imports the required libraries and confirms the CPU environment for OpenNyAI.

In [ ]:
import csv
import json
import sys
import time
from pathlib import Path

import torch
import spacy
from opennyai import Pipeline
from opennyai.utils import Data

if sys.version_info < (3, 13):
    raise RuntimeError(
        f"OpenNyAI 0.0.15 requires Python 3.13+. "
        f"Current: {sys.version.split()[0]}"
    )

spacy.require_cpu()
USE_GPU = False

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("OpenNyAI device: CPU")

### Load test judgments
Find the 100 test judgment files and prepare a separate folder for their tagged results.

In [ ]:
DATA_ROOT = Path("IN-Abs-prepared")
if not DATA_ROOT.exists() and (Path.cwd() / "IN-Abs-prepared").exists():
    DATA_ROOT = Path.cwd() / "IN-Abs-prepared"

judgment_dir = DATA_ROOT / "test" / "judgement"
if not judgment_dir.is_dir():
    judgment_dir = DATA_ROOT / "test" / "judgment"

if not judgment_dir.is_dir():
    raise FileNotFoundError(
        f"Could not find test/judgement under {DATA_ROOT.resolve()}. "
        "Put this notebook beside IN-Abs-prepared."
    )

OUTPUT_DIR = Path("opennyai_test_cpu_100")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSONL_PATH = OUTPUT_DIR / "tagged.jsonl"
CSV_PATH = OUTPUT_DIR / "sentence_roles.csv"

judgment_files = sorted(
    judgment_dir.glob("*.txt"),
    key=lambda p: (0, int(p.stem)) if p.stem.isdigit() else (1, p.stem.lower()),
)

records = []
for path in judgment_files:
    text = path.read_text(encoding="utf-8", errors="replace").strip()
    if text:
        records.append({
            "doc_id": path.stem,
            "source_file": str(path),
            "judgment": text,
        })

print("Judgment files found:", len(judgment_files))
print("Non-empty judgments found:", len(records))

if len(records) != 100:
    print(
        f"NOTE: expected 100 non-empty test judgments, found {len(records)}. "
        "Confirm DATA_ROOT points to the prepared IN-Abs dataset."
    )

if not records:
    raise ValueError("No non-empty judgment files found.")

records = records[:100]

print("Test judgments to tag:", len(records))
print("First IDs:", [record["doc_id"] for record in records[:5]])
print("Test output folder:", OUTPUT_DIR.resolve())


### Load pretrained tagger
Load OpenNyAI’s pretrained rhetorical-role model.

In [ ]:
role_pipeline = Pipeline(components=["Rhetorical_Role"], use_gpu=USE_GPU, verbose=True)
print("OpenNyAI rhetorical-role tagger is ready.")


In [ ]:
print("role_pipeline" in globals())

### Tag test judgments
Tag each test judgment, save sentence roles, and checkpoint progress so completed files can be skipped if rerun.

In [ ]:
TRIAL_DIR = Path("opennyai_test_cpu_100")
TRIAL_DIR.mkdir(parents=True, exist_ok=True)

TRIAL_JSONL_PATH = TRIAL_DIR / "tagged.jsonl"
TRIAL_CSV_PATH = TRIAL_DIR / "sentence_roles.csv"

trial_records = records[:100]
print("Test judgments in this run:", len(trial_records))


def make_json_safe(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, dict):
        return {str(k): make_json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_safe(v) for v in value]
    if hasattr(value, "to_dict"):
        return make_json_safe(value.to_dict())
    return str(value)


def extract_sentence_roles(prediction):
    output = []
    annotations = (
        prediction.get("annotations", [])
        if isinstance(prediction, dict)
        else []
    )

    for annotation in annotations:
        if not isinstance(annotation, dict):
            continue

        sentence = annotation.get("text") or annotation.get("sentence")
        labels = annotation.get("labels")
        if not sentence or labels is None:
            continue

        if isinstance(labels, (list, tuple)):
            role = ", ".join(
                str(label) for label in labels if label is not None
            )
        else:
            role = str(labels)

        if role:
            output.append({"sentence": sentence, "role": role})

    return output


completed_ids = set()

if TRIAL_JSONL_PATH.exists():
    with TRIAL_JSONL_PATH.open(encoding="utf-8") as input_file:
        for line in input_file:
            try:
                saved = json.loads(line)
                if saved.get("doc_id") is not None:
                    completed_ids.add(str(saved["doc_id"]))
            except json.JSONDecodeError:
                continue

todo = [
    record
    for record in trial_records
    if str(record["doc_id"]) not in completed_ids
]

print(f"Already completed: {len(completed_ids)} | Remaining: {len(todo)}")

started = time.perf_counter()

with TRIAL_JSONL_PATH.open("a", encoding="utf-8") as output_file:
    for offset, record in enumerate(todo, start=1):
        result = role_pipeline(
            Data([record["judgment"]], use_gpu=False)
        )

        prediction = (
            result[0]
            if isinstance(result, (list, tuple)) and result
            else result
        )
        prediction = make_json_safe(prediction)
        sentence_roles = extract_sentence_roles(prediction)

        tagged_judgment = "\n".join(
            f"[{item['role']}] {item['sentence']}"
            for item in sentence_roles
        )

        row = {
            "doc_id": str(record["doc_id"]),
            "source_file": record.get("source_file"),
            "prediction": prediction,
            "sentence_roles": sentence_roles,
            "tagged_judgment": tagged_judgment,
        }

        output_file.write(json.dumps(row, ensure_ascii=False) + "\n")
        output_file.flush()
        completed_ids.add(str(record["doc_id"]))

        if offset % 25 == 0 or offset == len(todo):
            elapsed = time.perf_counter() - started
            average = elapsed / offset
            remaining_seconds = average * (len(todo) - offset)

            print(
                f"{len(completed_ids)}/{len(trial_records)} completed | "
                f"latest judgment: {record['doc_id']} "
                f"({len(sentence_roles)} sentence tags) | "
                f"elapsed: {elapsed / 3600:.2f} h | "
                f"estimated remaining: {remaining_seconds / 3600:.2f} h"
            )


with TRIAL_CSV_PATH.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(
        csv_file,
        fieldnames=["doc_id", "sentence", "role"],
    )
    writer.writeheader()

    with TRIAL_JSONL_PATH.open(encoding="utf-8") as input_file:
        for line in input_file:
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue

            for item in row.get("sentence_roles", []):
                writer.writerow({
                    "doc_id": row.get("doc_id"),
                    "sentence": item.get("sentence"),
                    "role": item.get("role"),
                })

print(f"Finished: {len(completed_ids)} of {len(trial_records)} judgments")
print("JSONL checkpoint:", TRIAL_JSONL_PATH.resolve())
print("Sentence-role CSV:", TRIAL_CSV_PATH.resolve())

### Check tagging results
Counts the saved test judgments and sentence-role tags to confirm the output was written.

In [ ]:

with TRIAL_JSONL_PATH.open(encoding="utf-8") as f:
    saved_rows = [json.loads(line) for line in f if line.strip()]

print("Judgments saved:", len(saved_rows))
print(
    "Sentence-role tags saved:",
    sum(len(row.get("sentence_roles", [])) for row in saved_rows),
)
print("Example judgment ID:", saved_rows[0]["doc_id"] if saved_rows else "none")
print(
    "Example tags:",
    saved_rows[0].get("sentence_roles", [])[:3] if saved_rows else [],
)


In [ ]:
import json
from collections import Counter
from pathlib import Path

RESULT_PATH = Path(
    globals().get(
        "TRIAL_JSONL_PATH",
        "opennyai_test_cpu_100/tagged.jsonl",
    )
)

if not RESULT_PATH.is_file():
    raise FileNotFoundError(f"Tagged JSONL file not found: {RESULT_PATH.resolve()}")

rows = []
with RESULT_PATH.open(encoding="utf-8") as f:
    for line in f:
        if line.strip():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue

role_counts = Counter()
docs_with_role = Counter()

for row in rows:
    seen_roles_in_doc = set()

    for item in row.get("sentence_roles", []):
        raw_role = item.get("role", "")
        roles = [role.strip() for role in str(raw_role).split(",") if role.strip()]

        for role in roles:
            role_counts[role] += 1
            seen_roles_in_doc.add(role)

    for role in seen_roles_in_doc:
        docs_with_role[role] += 1

num_docs = len({str(row.get("doc_id", "unknown")) for row in rows})
num_sentences = sum(len(row.get("sentence_roles", [])) for row in rows)
total_assignments = sum(role_counts.values())

print("Tagged test judgments:", num_docs)
print("Tagged sentences:", num_sentences)
print("Role assignments:", total_assignments)
print("\nRole distribution:")

for role, count in role_counts.most_common():
    percent = 100 * count / total_assignments if total_assignments else 0
    doc_percent = 100 * docs_with_role[role] / num_docs if num_docs else 0
    print(
        f"{role:20s} {count:7d} tags ({percent:5.1f}%) | "
        f"present in {docs_with_role[role]:4d}/{num_docs} judgments "
        f"({doc_percent:5.1f}%)"
    )